<a href="https://colab.research.google.com/github/ashishsantikari/lab-sql-generation-with-transformer-api/blob/main/lab-sql-generation-with-transformer-api.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SQL Generation with Transformer API

In [1]:
!pip install torch transformers bitsandbytes accelerate sqlparse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 18.0 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
torch.cuda.is_available()

True

In [4]:
available_memory = torch.cuda.get_device_properties(0).total_memory

In [ ]:
print(available_memory)

15835660288

##Download the Model
Use any model on Colab (or any system with >30GB VRAM on your own machine) to load this in f16. If unavailable, use a GPU with minimum 8GB VRAM to load this in 8bit, or with minimum 5GB of VRAM to load in 4bit.

This step can take around 5 minutes the first time. So please be patient :)

In [5]:
model_name = "defog/sqlcoder-7b-2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
if available_memory > 15e9:
    # if you have atleast 15GB of GPU memory, run load the model in float16
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        device_map="auto",
        use_cache=True,
    )
else:
    # else, load in 8 bits – this is a bit slower
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        trust_remote_code=True,
        # torch_dtype=torch.float16,
        load_in_8bit=True,
        device_map="auto",
        use_cache=True,
    )

config.json:   0%|          | 0.00/691 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/515 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

##Set the Question & Prompt and Tokenize
Feel free to change the schema in the prompt below to your own schema

In [8]:
prompt = """### Task
Generate a SQL query to answer [QUESTION]{question}[/QUESTION]

### Instructions
- If you cannot answer the question with the available database schema, return 'I do not know'
- Remember that revenue is price multiplied by quantity
- Remember that cost is supply_price multiplied by quantity

### Database Schema
This query will run on a database whose schema is represented in this string:
CREATE TABLE products (
  product_id INTEGER PRIMARY KEY, -- Unique ID for each product
  name VARCHAR(50), -- Name of the product
  price DECIMAL(10,2), -- Price of each unit of the product
  quantity INTEGER  -- Current quantity in stock
);

CREATE TABLE customers (
   customer_id INTEGER PRIMARY KEY, -- Unique ID for each customer
   name VARCHAR(50), -- Name of the customer
   address VARCHAR(100) -- Mailing address of the customer
);

CREATE TABLE salespeople (
  salesperson_id INTEGER PRIMARY KEY, -- Unique ID for each salesperson
  name VARCHAR(50), -- Name of the salesperson
  region VARCHAR(50) -- Geographic sales region
);

CREATE TABLE sales (
  sale_id INTEGER PRIMARY KEY, -- Unique ID for each sale
  product_id INTEGER, -- ID of product sold
  customer_id INTEGER,  -- ID of customer who made purchase
  salesperson_id INTEGER, -- ID of salesperson who made the sale
  sale_date DATE, -- Date the sale occurred
  quantity INTEGER -- Quantity of product sold
);

CREATE TABLE product_suppliers (
  supplier_id INTEGER PRIMARY KEY, -- Unique ID for each supplier
  product_id INTEGER, -- Product ID supplied
  supply_price DECIMAL(10,2) -- Unit price charged by supplier
);

-- sales.product_id can be joined with products.product_id
-- sales.customer_id can be joined with customers.customer_id
-- sales.salesperson_id can be joined with salespeople.salesperson_id
-- product_suppliers.product_id can be joined with products.product_id

### Answer
Given the database schema, here is the SQL query that answers [QUESTION]{question}[/QUESTION]
[SQL]
"""

##Generate the SQL
This can be excruciatingly slow on a T4 in Colab, and can take 10-20 seconds per query. On faster GPUs, this will take ~1-2 seconds

Ideally, you should use `num_beams`=4 for best results. But because of memory constraints, we will stick to just 1 for now.

In [9]:
import sqlparse

def generate_query(question):
    updated_prompt = prompt.format(question=question)
    inputs = tokenizer(updated_prompt, return_tensors="pt").to("cuda")
    generated_ids = model.generate(
        **inputs,
        num_return_sequences=1,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.eos_token_id,
        max_new_tokens=400,
        do_sample=False,
        num_beams=1,
    )
    outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    torch.cuda.empty_cache()
    torch.cuda.synchronize()
    # empty cache so that you do generate more results w/o memory crashing
    # particularly important on Colab – memory management is much more straightforward
    # when running on an inference service
    return sqlparse.format(outputs[0].split("[SQL]")[-1], reindent=True)

In [10]:
question = "What was our revenue by product in the New York region last month?"
generated_sql = generate_query(question)

In [ ]:
print(generated_sql)


SELECT p.product_id,
       SUM(s.quantity * p.price) AS revenue
FROM sales s
JOIN salespeople sp ON s.salesperson_id = sp.salesperson_id
JOIN products p ON s.product_id = p.product_id
WHERE sp.region = 'New York'
  AND s.sale_date >= (CURRENT_DATE - INTERVAL '1 month')
GROUP BY p.product_id
ORDER BY revenue DESC NULLS LAST;


# Exercise
 - Complete the prompts similar to what we did in class.
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

In [12]:
question = "how many products are sold in the New York region last month?"
generated_sql = generate_query(question)
print(generated_sql)


SELECT SUM(s.quantity) AS total_products_sold
FROM sales s
JOIN salespeople sp ON s.salesperson_id = sp.salesperson_id
WHERE sp.region = 'New York'
  AND s.sale_date >= (CURRENT_DATE - INTERVAL '1 month');


In [14]:
question = "how has the sales performed QoQ in current FY?"
generated_sql = generate_query(question)
print(generated_sql)


SELECT (SUM(s.quantity) - SUM(lag(s.quantity, 1) OVER (
                                                       ORDER BY s.sale_date))) / NULLIF(lag(s.quantity, 1) OVER (
                                                                                                                 ORDER BY s.sale_date), 0) AS sales_to_sales_ratio
FROM sales s
WHERE EXTRACT(YEAR
              FROM s.sale_date) = EXTRACT(YEAR
                                          FROM CURRENT_DATE);


In [15]:
question = "how many product suppliers have less stock in last quarter?"
generated_sql = generate_query(question)
print(generated_sql)


SELECT COUNT(DISTINCT ps.supplier_id)
FROM product_suppliers ps
JOIN products p ON ps.product_id = p.product_id
WHERE p.quantity < 10
  AND EXTRACT(QUARTER
              FROM ps.last_updated) = EXTRACT(QUARTER
                                              FROM CURRENT_DATE) - 1;


## Summary of Learnings

This notebook demonstrates the process of setting up and using a large language model (LLM) for SQL generation. Specifically, we used the `defog/sqlcoder-7b-2` model from Hugging Face Transformers.

Here's a breakdown of the key takeaways:

1.  **Environment Setup**: We started by installing essential libraries such as `torch`, `transformers`, `bitsandbytes`, `accelerate`, and `sqlparse` to enable deep learning, model handling, quantization, and SQL formatting.
2.  **GPU Utilization**: We learned how to check for CUDA availability and available GPU memory, which is crucial for determining how to load the model efficiently (e.g., in `float16` for larger VRAM or `8-bit` quantization for more limited memory).
3.  **Model Loading**: We downloaded and loaded the `sqlcoder-7b-2` model using `AutoTokenizer` and `AutoModelForCausalLM`, dynamically adjusting the loading precision (`torch_dtype=torch.float16` or `load_in_8bit=True`) based on available GPU memory.
4.  **Prompt Engineering for SQL Generation**: A detailed prompt template was designed, incorporating:
    *   A clear `Task` to generate SQL from a natural language `QUESTION`.
    *   Specific `Instructions` for handling unknown questions, calculating revenue and cost, and using the provided schema.
    *   A comprehensive `Database Schema` definition, including table structures, column types, and relationships, which is vital for the model to generate accurate SQL.
5.  **SQL Generation Functionality**: A `generate_query` function was implemented. This function:
    *   Formats the prompt with the user's question.
    *   Tokenizes the input for the LLM.
    *   Uses the loaded model to generate SQL queries, with parameters like `num_return_sequences`, `max_new_tokens`, and `num_beams` (though `num_beams=1` was used due to memory constraints).
    *   Clears the CUDA cache after generation to manage GPU memory effectively.
    *   Formats the generated SQL using `sqlparse` for readability.
6.  **Practical Application**: We demonstrated the model's capability by generating SQL queries for various natural language questions, such as "What was our revenue by product in the New York region last month?", "how many products are sold in the New York region last month?", "how has the sales performed QoQ in current FY?", and "how many product suppliers have less stock in last quarter?".

Overall, the notebook provided a hands-on experience in using a transformer-based LLM for text-to-SQL generation, highlighting the importance of appropriate model loading, detailed prompt engineering, and efficient memory management in a Colab environment.